# Compute, Access, and When Not to Use Any of This

**Computational Text Analysis Pack · Module 28 — The Neural Critique**

*Technique:* Cost and carbon; the stochastic-parrots argument; a decision procedure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/15-the-neural-critique/28b_cost_access_and_refusal.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The last lesson in the neural arc, and it argues against the arc.

Three critiques the survey treats as live, and a decision procedure that takes
them seriously without pretending the tools are useless.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

## 1. Compute and carbon

Strubell et al. (2019) put the training of one large NLP model, with
hyperparameter search, at hundreds of thousands of pounds of CO2. Costs have
fallen per unit of compute and total compute has risen faster.

The DH-relevant point is not guilt. It is that **compute is a form of access**,
and access is unevenly distributed in exactly the way this course's subject
matter is.

In [ ]:
import numpy as np

# Orders of magnitude, from published figures. Rounded, and the point survives.
COSTS = [
    ("TF-IDF + logistic regression (this course)", 0.00002, "laptop, seconds"),
    ("Frozen LaBSE features, 1,400 docs", 0.002, "one GPU-minute"),
    ("Fine-tune BERT-base, small corpus", 2.0, "a few GPU-hours"),
    ("Pretrain AfriBERTa", 5_000, "days on a GPU cluster"),
    ("Pretrain a frontier LLM", 5_000_000, "months on thousands of GPUs"),
]
print(f"{'workload':<44}{'approx GBP':>13}   scale")
print("-" * 86)
for name, gbp, scale in COSTS:
    print(f"{name:<44}{gbp:>13,.5f}   {scale}")
print()
base = COSTS[0][1]
print(f"The last row is roughly {COSTS[-1][1]/base:,.0f}x the first.")
print("A researcher without funding is not doing the last two rows, ever. The")
print("first two are open to anyone with a laptop — which is why this course")
print("spends 22 modules there and 6 here.")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh([c[0][:38] for c in COSTS][::-1], [c[1] for c in COSTS][::-1], color="steelblue")
ax.set_xscale("log"); ax.set_xlabel("approximate cost, GBP (log scale)")
ax.set_title("Access is the real constraint")
plt.tight_layout()

## 2. Stochastic parrots, and whose language is in the model

Bender, Gebru, McMillan-Major and Mitchell (2021) made the argument that has
shaped the field's self-criticism: large language models are trained on
uncurated web text, they encode its biases, they are environmentally costly, and
their fluency invites us to attribute understanding that is not there.

For African DH the sharpest edge is **coverage**. Module 23a measured it: every
model covers well under 1% of Africa's languages, and the ones it covers best
are those with the most colonial-era digitisation.

In [ ]:
# Data provenance: a question you can ask of any model, and usually cannot answer.
QUESTIONS = [
    "What text was this model trained on?",
    "Were the speakers or writers asked?",
    "Which languages, and in what proportion?",
    "Can the training data be inspected?",
    "Can a community request removal of its texts?",
]
STATUS = {
    "MasakhaNER / MasakhaNEWS": ["yes - documented", "community-built", "yes, published",
                                 "yes", "yes - governed by Masakhane"],
    "AfriBERTa": ["partly - listed sources", "no", "yes, 11 languages", "partly", "no"],
    "A frontier LLM": ["no", "no", "no", "no", "no"],
}
print(f"{'question':<44}" + "".join(f"{k[:22]:>24}" for k in STATUS))
print("-" * 116)
for i, q in enumerate(QUESTIONS):
    print(f"{q:<44}" + "".join(f"{v[i][:22]:>24}" for v in STATUS.values()))
print()
print("The left-hand column is what a humanities scholar would ask of any")
print("archive. The right-hand column is the answer for most of what this")
print("module taught you to use.")

## 3. The decision procedure

Run these in order. Stop at the first that answers your question.

1. **Can you count it?** Frequency, keyness, collocation, dispersion. Cheap,
   interpretable, reproducible. Modules 2–5.
2. **Can a rule express it?** Regex, a gazetteer, a finite-state grammar.
   Auditable, and better than statistics on small or morphologically rich data.
   Module 15.
3. **Do you have labels?** Classical supervised learning with a linear model,
   whose coefficients you can read. Module 7.
4. **Are you crossing languages, or short of labels?** Pretrained embeddings,
   frozen. Module 23.
5. **Is the domain far from pretraining, with thousands of labels?** Fine-tune.
   Module 26.
6. **Do you need generated text?** Ground it in a corpus you control, and check
   every citation. Module 27.

Most DH questions stop at step 1 or 2. That is not a failure of ambition; it is
what it looks like to match the method to the question.

In [ ]:
# One last measurement, as a summary of the whole course.
LADDER = [
    ("count words (M2)", 0.0, "instant", "total"),
    ("TF-IDF + logistic (M7)", 0.811, "seconds", "high - coefficients"),
    ("Keras embedding-average (M13)", 0.771, "seconds", "low"),
    ("frozen LaBSE + logistic (M23)", None, "minutes", "low"),
    ("fine-tuned transformer (M26)", None, "hours + GPU", "very low"),
    ("prompted LLM (M27)", None, "per call, per token", "none"),
]
print(f"{'method':<34}{'accuracy':>12}{'cost':>22}{'interpretability':>20}")
print("-" * 90)
for name, acc, cost, interp in LADDER:
    a = "-" if acc in (None, 0.0) else f"{acc:.3f}"
    print(f"{name:<34}{a:>12}{cost:>22}{interp:>20}")
print()
print("Accuracy rises down the table. So does cost, and interpretability falls")
print("off a cliff. There is no row that is simply best — which is why the")
print("question, not the fashion, has to pick the row.")

### 🔧 Your turn 1

Take the research question you brought to this course and walk it down the six steps. Where does it stop?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

If it stops at step 1 or 2, you have saved yourself months and produced
something a reader can check line by line — and you should say in your methods
section that you tested the more elaborate options and did not need them, because
that sentence is itself a finding. If it genuinely reaches step 5 or 6, you now
know what you are paying: money, reproducibility, and the ability to explain your
own result. Paying that knowingly is scholarship. Paying it because the tool was
fashionable is not.

</details>